# 08 — Method comparison

The 23 Phase 1 backtests compared under the design pre-registered in §1: reproduction gates, performance, bootstrap Sharpe intervals, Sharpe-ratio tests against EW (H1) and CAPM α with a multiple-testing adjustment (H2). 60/40 appears in the descriptive tables and figures only.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, the panel the classes read, and the SPY excess return used as the CAPM market factor.

In [ ]:
import math
import sys
from itertools import combinations
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb08_helpers as h
from maplab import Panel, apply_style

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

F4 = lambda v: f"{v:.4f}"
E3 = lambda v: f"{v:.3e}"

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

In [ ]:
reg = ml.inference.load_registration("nb08")
ml.inference.render_registration(reg)

## §2 The statistics

General forms of the estimators named in §1. $x_t$ is a run's daily net return in excess of BIL and $T$ the number of days in a window; the Sharpe ratio is $\widehat{SR} = \sqrt{252}\,\bar x / s_x$ with the sample mean and standard deviation.

**Newey–West long-run covariance** (Newey & West 1987). For a vector series $y_t$ with autocovariances $\hat\Gamma_j = T^{-1}\sum_{t=j+1}^{T}(y_t-\bar y)(y_{t-j}-\bar y)'$,
$$\hat\Psi = \hat\Gamma_0 + \sum_{j=1}^{L}\Big(1-\frac{j}{L+1}\Big)\big(\hat\Gamma_j+\hat\Gamma_j'\big).$$
The Bartlett weights keep $\hat\Psi$ positive semi-definite; the lag $L = \lfloor 4(T/100)^{2/9}\rfloor$ is the rule of thumb of Newey & West (1994).

**Sharpe-ratio difference** (Jobson & Korkie 1981; Memmel 2003; HAC form as in Ledoit & Wolf 2008). With $\theta = (\mu_a, \mu_b, \gamma_a, \gamma_b)$, $\gamma = E[x^2]$ and $\sigma = \sqrt{\gamma-\mu^2}$, the difference $\Delta = \mu_a/\sigma_a - \mu_b/\sigma_b$ has
$$\nabla\Delta = \Big(\frac{\gamma_a}{\sigma_a^3},\; -\frac{\gamma_b}{\sigma_b^3},\; -\frac{\mu_a}{2\sigma_a^3},\; \frac{\mu_b}{2\sigma_b^3}\Big)', \qquad \mathrm{se}(\hat\Delta) = \sqrt{\nabla\Delta'\,\hat\Psi\,\nabla\Delta / T},$$
with $\hat\Psi$ the Newey–West covariance of $y_t = (x_{a,t}, x_{b,t}, x_{a,t}^2, x_{b,t}^2)$. Both are annualized by $\sqrt{252}$; $t = \hat\Delta/\mathrm{se}$ with a two-sided normal p. The moments divide by $T$, so $\hat\Delta$ differs from the difference of the $s_x$-based Sharpe ratios by the ddof gap printed in §6.

**CAPM α.** $x_t = \alpha + \beta\,m_t + \varepsilon_t$, with $m_t$ the SPY return in excess of BIL. The coefficient covariance is $(X'X)^{-1}\,T\hat S\,(X'X)^{-1}$, with $\hat S$ the Newey–West long-run covariance of $X_t\hat\varepsilon_t$ (not demeaned, no degrees-of-freedom correction); α is annualized by 252.

**Stationary bootstrap** (Politis & Romano 1994). A resample of length $T$ is built from blocks that start at a uniform index and end each day with probability $p = 1/21$, so block lengths are geometric with mean 21, wrapping at the end of the sample. One index set per resample is applied to every series, which keeps their cross-correlation. Percentile intervals take the 2.5% and 97.5% quantiles of the resampled statistic (Efron & Tibshirani 1993); the bootstrap p of $\hat\Delta$ is the share of resamples with $\lvert\Delta^*-\hat\Delta\rvert \ge \lvert\hat\Delta\rvert$.

**Multiple testing.** For $m$ raw p-values ordered $p_{(1)} \le \dots \le p_{(m)}$, Holm (1979) sets $\tilde p_{(j)} = \max_{i\le j}\min\{1,(m-i+1)\,p_{(i)}\}$ and controls the family-wise error rate under any dependence; Benjamini & Hochberg (1995) set $\tilde p_{(j)} = \min_{i\ge j}\min\{1, m\,p_{(i)}/i\}$ and control the false discovery rate under independence or positive dependence (Benjamini & Yekutieli 2001). Romano & Wolf (2005) step down on the bootstrap distribution of the maximum: with $\lvert t\rvert$ ordered from the largest,
$$\tilde p_{(j)} = \max_{i\le j}\; \frac{1}{B}\sum_{b=1}^{B} \mathbf 1\Big\{\max_{k\ge i}\lvert t^*_{b,(k)}\rvert \ge \lvert t_{(i)}\rvert\Big\}, \qquad t^*_b = (\hat\alpha^*_b-\hat\alpha)/\mathrm{se}_{NW},$$
so it accounts for the dependence among the tests that Holm ignores.

## §3 Runs and reproduction gates

Every α-family run is rebuilt with the Phase 1 constructors and harness, with 60/40 alongside as the second benchmark. The Sharpe ratios and OLS CAPM α t-statistics must match the Phase 1 notebooks, or the notebook stops.

In [ ]:
CORE, REF, ALPHA = h.CORE, h.REF, h.ALPHA_FAMILY
BENCH = h.BENCH

results = h.build_runs(simple_returns, panel)
bench = h.build_benchmarks(simple_returns, panel)
all_runs = {**results, **bench}
assert len(all_runs) == len(results) + len(bench)
for name, r in all_runs.items():
    assert r["wlog"].index.equals(rdates), f"{name}: wlog index != rdates"
print(f"{len(results)} α-family runs completed: {', '.join(results)}")
print(f"benchmark: {', '.join(bench)}")

Summary table for every run in the three windows and the Sharpe reproduction gate against `registrations/reproduction.toml`, 60/40 included (notebook 01).

In [ ]:
summary_table = h.summary_table(results, ALPHA, split_ts, rf_daily)
ml.inference.check_reproduction(summary_table, h.REPRO_KEYS)
bench_summary = h.summary_table(bench, BENCH, split_ts, rf_daily)
ml.inference.check_reproduction(bench_summary, h.BENCH_REPRO_KEYS)
print(f"REPRODUCTION GATE: PASSED -- Sharpe to 2 dp for all {len(ALPHA)} runs and 60/40, all three windows.")

OLS CAPM regression of each run's excess return on the market excess return, and the α t-statistic gate against the Phase 1 printed values.

In [ ]:
capm_ols = ml.inference.capm_table(results, ALPHA, MKT_simple, rf_daily)
print(capm_ols.to_string(float_format=F4))

T_ALPHA_PHASE1 = {"GMV(S)": 1.66, "MaxSharpe(S)": 2.08, "MDP(S)": 2.02, "HRP(S)": 2.02,
                  "HRP(LW)": 1.99, "HRP[positional]": 1.92, "HRP[ward]": 1.93}
for name, exp in T_ALPHA_PHASE1.items():
    got = round(float(capm_ols.loc[name, "t_alpha"]), 2)
    assert got == exp, f"OLS ALPHA GATE FAILED: {name} t_alpha {got} != {exp}"
print("GATES PASS")

Daily excess returns (net − BIL) for every run on the scoring window, split into the summary table's three windows.

In [ ]:
X = h.excess_frame(results, ALPHA, rf_daily)
XW = h.window_frames(X, split_ts)
X_full, X_test = XW["full"], XW["test (> split)"]

for w, Xw in XW.items():
    print(f"{w:<18} T={len(Xw):>5}  first={Xw.index[0].date()}  last={Xw.index[-1].date()}")
assert [len(Xw) for Xw in XW.values()] == [4338, 3504, 834]
assert X_full.index[0] == results["EW"]["net"].index.min()

## §4 Performance (descriptive)

The eight core runs, the two references and 60/40 in the three windows. 60/40 is not in the §1 registration: it was added after the registration was frozen, and its numbers were first printed in notebook 01. Every 60/40 comparison below is exploratory and enters no registered test.

In [ ]:
PERF_COLS = ["ann_return", "ann_vol", "sharpe", "max_dd", "ann_turnover"]
perf = pd.concat([summary_table.loc[CORE + REF], bench_summary])
assert list(perf.index.get_level_values("strategy").unique()) == CORE + REF + BENCH
print(perf[PERF_COLS].to_string(float_format=F4))

60/40 has the highest full-window Sharpe ratio in the table (0.91); BMV(0.3) is the highest core run (0.86). In the test window MaxSharpe(S) and BMV(0.3) are above it (1.13, 1.13 vs 1.00). Among the core runs, turnover runs from 14%/yr (EW) to 248%/yr (MaxSharpe(S)).

Maximum drawdown of each core run and 60/40 on the full window, with its peak, trough and recovery dates.

In [ ]:
dd_dates = h.drawdown_dates(all_runs, CORE + BENCH)
print(dd_dates.to_string(float_format=F4))

### Figure F1/F2 — wealth and drawdown of the core runs and 60/40, with BIL wealth; the dotted line marks the train/test split.

In [ ]:
WD = CORE + BENCH
fig = ml.plotting.wealth_drawdown({m: all_runs[m]["net"] for m in WD}, rf_daily, split_ts)
plt.show()

ax_w, ax_d = fig.axes[0], fig.axes[1]
wline = {l.get_label(): l for l in ax_w.get_lines()}
dline = {l.get_label(): l for l in ax_d.get_lines()}
term_diff = max(abs(float((1.0 + all_runs[m]["net"]).prod()) - float(wline[m].get_ydata()[-1])) for m in WD)
dd_diff = max(abs(float(np.min(dline[m].get_ydata())) / 100.0 - float(perf.loc[(m, "full"), "max_dd"]))
              for m in WD)
print(f"terminal wealth, (1+net).prod() vs plotted last value: max abs diff = {term_diff}")
print(f"min drawdown, plotted vs summary max_dd: max abs diff = {E3(dd_diff)}")
assert term_diff == 0.0
assert dd_diff <= 1e-12
assert wline["60/40"].get_color() == ml.plotting.METHOD_COLORS["60/40"]

Six of the eight core runs have their maximum drawdown in March 2020; BL(0.1) bottoms on 2023-10-03 and MDP(S) on 2016-01-19. 60/40 ends with the highest wealth of the nine runs and has the deepest drawdown (−21.3%, trough 2022-10-14).

Active return, tracking error and correlation of each core run against EW and against 60/40, from daily net returns; active return and tracking error also on the test window. Descriptive only: no test is attached.

In [ ]:
active = h.active_stats(all_runs, CORE, ["EW"] + BENCH, split_ts)
print(active.to_string(float_format=F4))

P1_ACTIVE_VS_EW = {"BL(0.1,S)": -1.36, "MDP(S)": -3.17, "ERC(S)": -2.78, "HRP(S)": -3.30}
for m, exp in P1_ACTIVE_VS_EW.items():
    got = round(100 * float(active.loc[("EW", m), "active_ret"]), 2)
    assert got == exp, f"ACTIVE CROSS-CHECK FAILED: {m} vs EW {got} != {exp} %/yr"
print("active return vs EW matches the Phase 1 paired means (notebooks 03, 05, 06, 07)")

Against EW, tracking error runs from 3.2%/yr (BL(0.1)) to 7.9%/yr (MaxSharpe(S)); against 60/40, from 5.0%/yr (EW) to 10.0%/yr (MaxSharpe(S)). Six of the seven non-EW core runs are more correlated with EW than with 60/40; the exception is BMV(0.3) (0.94 with 60/40, 0.86 with EW). Every core run trails 60/40 on the full window, by 3.39 to 7.21%/yr.

## §5 Bootstrap Sharpe intervals (descriptive)

Stationary-bootstrap Sharpe ratios of the core and reference runs, one shared index draw per window, with percentile intervals.

In [ ]:
B, BLOCK, SEED = 10000, 21, 20260923
CR = CORE + REF

sr_star = {}
ci_rows = []
sr_gap = {}
for w, Xw in XW.items():
    idx = list(ml.robust.bootstrap_index_chunks(len(Xw), B, BLOCK, seed=SEED))
    draws = ml.robust.bootstrap_sharpe(Xw[CR], idx)
    if w == "full":
        IDX_full = idx  # shared with §7
    del idx
    assert draws.shape == (B, len(CR))
    sr_star[w] = pd.DataFrame(draws, columns=CR)
    sr_point = Xw[CR].mean() / Xw[CR].std(ddof=1) * np.sqrt(ml.TRADING_DAYS)
    sr_gap[w] = (sr_point - summary_table.xs(w, level="window").loc[CR, "sharpe"]).abs()
    lo, hi = ml.robust.percentile_ci(draws, 0.95)
    for j, m in enumerate(CR):
        ci_rows.append({"label": m, "window": w, "SR": float(sr_point[m]), "lo": float(lo[j]), "hi": float(hi[j])})

sr_gap = pd.DataFrame(sr_gap)
print("|point SR (ddof=1) - summary Sharpe| by window, max over runs:")
print(sr_gap.max().to_string(float_format=lambda v: f"{v:.3e}"))
assert float(sr_gap.max().max()) <= 1e-10

ci_long = pd.DataFrame(ci_rows)
SR_full_star = sr_star["full"]

Point Sharpe ratio and percentile interval per run and window.

In [ ]:
ci_table = ci_long.pivot(index="label", columns="window", values=["SR", "lo", "hi"])
ci_table = ci_table.swaplevel(0, 1, axis=1)
ci_table = ci_table.reindex(index=CR, columns=pd.MultiIndex.from_product([list(XW), ["SR", "lo", "hi"]]))
print(ci_table.to_string(float_format=F4))

### Figure F3 — bootstrap Sharpe intervals by run and window.

In [ ]:
fig = ml.plotting.forest_plot(ci_long, "label", "SR", "lo", "hi", group="window", ref=0.0,
                              xlabel="Sharpe vs BIL")
plt.show()

Every full-window interval excludes 0, with lower bounds from 0.13 (IVP) to 0.41 (BMV(0.3)). In the 3.3-year test window 7 of the 10 lower bounds are below 0.

## §6 H1 — Sharpe vs EW (registered) and all core pairs (descriptive)

Delta-method Newey–West test of each core method's Sharpe ratio against EW on the full window, Holm-adjusted, with the test-window difference and a bootstrap p as a cross-check.

In [ ]:
H1_KEYS = {"GMV(S)": "GMV", "MaxSharpe(S)": "MaxSharpe", "BMV(0.3)": "BMV", "BL(0.1,S)": "BL",
           "MDP(S)": "MDP", "ERC(S)": "ERC", "HRP(S)": "HRP"}
H1_METHODS = [m for m in CORE if m != "EW"]
assert H1_METHODS == list(H1_KEYS)

sr_full = X_full[CORE].mean() / X_full[CORE].std(ddof=1) * np.sqrt(ml.TRADING_DAYS)

rows = []
for m in H1_METHODS:
    r = ml.robust.sharpe_diff_hac(X_full[m], X_full["EW"])
    assert r["L"] == 9
    rt = ml.robust.sharpe_diff_hac(X_test[m], X_test["EW"])
    assert rt["L"] == 6
    d_hat = float(sr_full[m] - sr_full["EW"])
    d_star = (SR_full_star[m] - SR_full_star["EW"]).to_numpy()
    rows.append({"method": m, "dSR_full": r["d_sr"], "se": r["se"], "t": r["t"], "p_raw": r["p"],
                 "dSR_test": rt["d_sr"], "same_sign_test": bool(np.sign(rt["d_sr"]) == np.sign(r["d_sr"])),
                 "boot_p": float(ml.robust.bootstrap_p(d_hat, d_star)), "d_hat_ddof1": d_hat})

h1 = pd.DataFrame(rows).set_index("method")
h1["p_holm"] = ml.robust.holm(h1["p_raw"].to_numpy())

ddof_gap = float((h1["dSR_full"] - h1["d_hat_ddof1"]).abs().max())
print(f"registered ddof gap: max |d_sr (delta, ddof 0) - d_hat (ddof 1)| over the 7 pairs = {E3(ddof_gap)}")

H1 table: full-window ΔSR vs EW with Newey–West standard error, raw and Holm p, test-window ΔSR and sign agreement, and the bootstrap p.

In [ ]:
h1_table = h1[["dSR_full", "se", "t", "p_raw", "p_holm", "dSR_test", "same_sign_test", "boot_p"]]
print(h1_table.to_string(float_format=F4))

h1_stats = {}
for m, key in H1_KEYS.items():
    row = h1.loc[m]
    h1_stats[f"H1_{key}"] = dict(
        display=f"ΔSR {row['dSR_full']:+.2f}, t {row['t']:.2f}, Holm p {row['p_holm']:.3f}",
        p_adj=float(row["p_holm"]), same_sign_test=bool(row["same_sign_test"]),
    )
assert set(h1_stats) == {x["id"] for x in reg["hypothesis"] if x["id"].startswith("H1_") and x["kind"] == "adjusted_null"}

ΔSR against EW runs from −0.11 (GMV(S)) to +0.12 (BMV(0.3)); the largest absolute t is 1.01, and every Holm-adjusted p is 1.000. The standard error is smallest for BL(0.1) (0.10) and largest for MaxSharpe(S) (0.26), the runs with the smallest and largest tracking error against EW in §4; an unadjusted 5% rejection needs a Sharpe gap of about 0.20 to 0.50. The bootstrap p agrees: the smallest is 0.323.

H1-all: the same test for every pair of core runs on the full window, with Benjamini–Hochberg adjustment over the pairs.

In [ ]:
T_mat = pd.DataFrame(np.nan, index=CORE, columns=CORE)
P_mat = pd.DataFrame(np.nan, index=CORE, columns=CORE)
pair_rows = []
for a, b in combinations(CORE, 2):
    r = ml.robust.sharpe_diff_hac(X_full[a], X_full[b])
    assert r["L"] == 9
    T_mat.loc[a, b], T_mat.loc[b, a] = r["t"], -r["t"]
    P_mat.loc[a, b] = P_mat.loc[b, a] = r["p"]
    pair_rows.append({"pair": f"{a} - {b}", "dSR": r["d_sr"], "t": r["t"], "p_raw": r["p"]})

pairs_all = pd.DataFrame(pair_rows).set_index("pair")
assert len(pairs_all) == 28
pairs_all["p_bh"] = ml.robust.bh(pairs_all["p_raw"].to_numpy())
assert np.allclose(T_mat.to_numpy(), -T_mat.to_numpy().T, equal_nan=True)

print("t (row - column):")
print(T_mat.to_string(float_format=lambda v: f"{v:.2f}"))
print("\nraw p:")
print(P_mat.to_string(float_format=F4))

sig = pairs_all[pairs_all["p_bh"] < 0.10]
print("\npairs with BH p < 0.10:")
print(sig.to_string(float_format=F4) if len(sig) else "none")

ew_col_diff = float((T_mat.loc[H1_METHODS, "EW"] - h1["t"]).abs().max())
print(f"\nEW column of T vs H1 table t: max abs diff = {E3(ew_col_diff)}")
assert ew_col_diff <= 1e-12

### Figure F4 — ΔSR t-statistic for every core pair (row − column), full window.

In [ ]:
fig = ml.plotting.pair_matrix(T_mat, CORE, "ΔSR t (row − column), full window, Newey–West")
plt.show()

Across the 28 core pairs the largest absolute t is 1.50 (BMV(0.3) − BL(0.1)), and Benjamini–Hochberg at q = 0.10 flags no pair.

## §7 H2 — CAPM α, Romano–Wolf adjusted

CAPM regression of each α-family run on the market excess return over the full window, with Newey–West standard errors.

In [ ]:
Y_cols, x_ref = {}, None
for m in ALPHA:
    net = results[m]["net"]
    idx_m = net.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    Y_cols[m] = net.loc[idx_m] - rf_daily.loc[idx_m]
    x_m = MKT_simple.loc[idx_m]
    if x_ref is None:
        x_ref = x_m
    assert x_m.index.equals(x_ref.index) and np.array_equal(x_m.to_numpy(), x_ref.to_numpy())
Y = pd.DataFrame(Y_cols)[ALPHA]
x = x_ref
x_df = x.to_frame("MKT")
assert Y.index.equals(X_full.index) and Y.shape == (4338, len(ALPHA))

alpha_hat, se_nw = pd.Series(np.nan, index=ALPHA), pd.Series(np.nan, index=ALPHA)
for m in ALPHA:
    fit = ml.robust.ols_hac(Y[m], x_df, L=None)
    assert fit["L"] == 9
    alpha_hat[m] = float(fit["coef"]["alpha"]) * ml.TRADING_DAYS
    se_nw[m] = float(fit["se"]["alpha"]) * ml.TRADING_DAYS
t_nw = alpha_hat / se_nw

alpha_gap = float((alpha_hat - capm_ols.loc[ALPHA, "alpha_ann"]).abs().max())
print(f"max |alpha_hat - capm_table alpha_ann| = {E3(alpha_gap)}")
assert alpha_gap <= 1e-12

Stationary-bootstrap CAPM α on the shared full-window indices from §5, centred and studentized by the Newey–West se, and the Romano–Wolf step-down adjusted p.

In [ ]:
assert sum(c.shape[0] for c in IDX_full) == B and all(c.shape[1] == len(X_full) == 4338 for c in IDX_full)
alpha_star = ml.robust.bootstrap_capm_alpha(Y.to_numpy(), x.to_numpy(), IDX_full)
assert alpha_star.shape == (B, len(ALPHA))
t_star = (alpha_star - alpha_hat.to_numpy()) / se_nw.to_numpy()
p_rw = pd.Series(ml.robust.romano_wolf(t_nw.to_numpy(), t_star), index=ALPHA)

sd_t = t_star.std(axis=0, ddof=1)
print(f"sd of t* per run: min {np.min(sd_t):.3f}  median {np.median(sd_t):.3f}  max {np.max(sd_t):.3f}")

Test-window OLS α sign for each run, and the descriptive raw, Holm and BH p alongside the registered Romano–Wolf p.

In [ ]:
test_sel = Y.index > split_ts
alpha_test = pd.Series({m: float(ml.inference.ols(Y.loc[test_sel, m], x_df.loc[test_sel])["coef"]["alpha"])
                        for m in ALPHA})
same_sign = np.sign(alpha_test) == np.sign(alpha_hat)

CLUSTER_ALIAS = {"HRP[positional](S)": "HRP[positional]", "HRP[ward](S)": "HRP[ward]"}
h2_ids, cluster = {}, pd.Series(False, index=ALPHA)
for hyp in reg["hypothesis"]:
    if not (hyp["id"].startswith("H2_") and hyp["kind"] == "adjusted_null"):
        continue
    lab = hyp["label"]
    is_cluster = lab.endswith(" [cluster]")
    name = lab.removesuffix(" [cluster]").removeprefix("H2 (α ").removesuffix(" = 0)")
    name = CLUSTER_ALIAS.get(name, name)
    assert name in ALPHA and name not in h2_ids, name
    h2_ids[name] = hyp["id"]
    cluster[name] = is_cluster
assert set(h2_ids) == set(ALPHA) and len(set(h2_ids.values())) == len(ALPHA)
assert sorted(cluster[cluster].index) == sorted(["MaxSharpe(S)", "MaxSharpe(LW)", "MaxSharpe(OAS)", "MDP(S)", "MDP(LW)",
                                                 "HRP(S)", "HRP(LW)", "HRP[positional]", "HRP[ward]"])

p_raw = pd.Series([math.erfc(abs(t) / math.sqrt(2)) for t in t_nw], index=ALPHA)
h2_table = pd.DataFrame({
    "alpha_ann": alpha_hat, "se_NW": se_nw, "OLS_t": capm_ols.loc[ALPHA, "t_alpha"], "NW_t": t_nw,
    "p_raw": p_raw, "p_holm": ml.robust.holm(p_raw.to_numpy()), "p_bh": ml.robust.bh(p_raw.to_numpy()),
    "p_rw": p_rw, "alpha_test_sign": np.sign(alpha_test).astype(int), "same_sign_test": same_sign,
    "cluster": cluster,
}).loc[ALPHA]
print(h2_table.to_string(float_format=F4))

h2_stats = {}
for m in ALPHA:
    a, t, p = float(alpha_hat[m]), float(t_nw[m]), float(p_rw[m])
    h2_stats[h2_ids[m]] = dict(display=f"α {a:+.2%}, NW t {t:.2f}, RW p {p:.3f}",
                               p_adj=p, same_sign_test=bool(same_sign[m]))

### Figure F5 — CAPM α with a Newey–West interval per α-family run.

In [ ]:
f5 = pd.DataFrame({
    "label": [m + (" *" if cluster[m] else "") for m in ALPHA],
    "alpha": 100 * alpha_hat.to_numpy(),
    "lo": 100 * (alpha_hat - 1.96 * se_nw).to_numpy(),
    "hi": 100 * (alpha_hat + 1.96 * se_nw).to_numpy(),
})
fig = ml.plotting.forest_plot(f5, "label", "alpha", "lo", "hi", ref=0.0,
                              xlabel="CAPM α, %/yr (± 1.96 Newey–West se)",
                              title="CAPM α, full window (* = pre-labelled cluster)")
plt.show()

excl = ((f5["lo"] > 0) | (f5["hi"] < 0)).to_numpy()
print("interval excludes 0 iff |NW t| >= 1.96, all runs:", bool(np.all(excl == (t_nw.abs() >= 1.96).to_numpy())))

Newey–West standard errors split the pre-labelled cluster. The four HRP variants fall from OLS t 1.92–2.02 to Newey–West t 1.72–1.84; MaxSharpe (S, LW, OAS) and MDP(S) stay above 2 (Newey–West t 2.04–2.16, raw p 0.031–0.041). None survives an adjustment: Holm p 0.72–0.83, BH p 0.197, Romano–Wolf p 0.12–0.14; the smallest Romano–Wolf p in the family is 0.124 (MaxSharpe(LW)). Romano–Wolf gives the smallest adjusted p of the three because it uses the joint bootstrap distribution of the 23 correlated statistics. The Phase 1 cluster of OLS t near 2 is consistent with 23 correlated tests and no α.

## Verdict table

The §1 rules applied mechanically to the H1 statistics in §6 and the H2 statistics in §7.

In [ ]:
table = ml.inference.evaluate(reg, {**h1_stats, **h2_stats})
assert len(table) == 30
print(table.to_string(index=False))
print()
print(table["verdict"].value_counts().to_string())

## §8 TAKEAWAY — 30 of 30 registered nulls hold

- **H1: no core method's Sharpe ratio differs from EW's.** ΔSR −0.11 to +0.12, largest absolute t 1.01, every Holm-adjusted p is 1.000. Across the 28 core pairs the largest absolute t is 1.50 and Benjamini–Hochberg flags no pair.
- **H2: no CAPM α survives the search over 23 runs.** The four runs with Newey–West t above 2 (MaxSharpe ×3, MDP(S)) have Romano–Wolf p 0.12–0.14; the smallest in the family is 0.124.
- **Power is low.** An unadjusted 5% rejection against EW needs a Sharpe gap of about 0.20 to 0.50; NULL HOLDS is not evidence of equal Sharpe ratios.
- **Descriptive.** Every full-window Sharpe interval excludes 0; 7 of the 10 test-window lower bounds are below 0. Six of the eight core runs bottom in March 2020.
- **60/40 (exploratory, not tested).** Highest full-window Sharpe (0.91) and ending wealth, deepest drawdown (−21.3%); every core run trails it by 3.39 to 7.21%/yr.

The Phase 1 methods cannot be told apart from EW, from each other or from zero α on this sample; a case for any of them has to rest on something other than full-sample Sharpe, such as the regime behaviour taken up in Phase 3.